In [2]:
"""
거래처 계층 구조 테이블 생성
- 슬래시(/) 앞 = Parent (기업 본사/주관사)
- 슬래시(/) 뒤 = Child (지점/납품처/하위거래처)
- 단독 행 = 슬래시 없는 독립 거래처 (Parent = None)
- 슬래시 패턴에만 등장하는 Parent → 자동으로 Parent 행 생성
출력: 하나의 Master 테이블에서 parent_id로 계층 조회 가능
"""
import msoffcrypto
import io

import pandas as pd
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

# ─────────────────────────────────────────────
# 0. 파일 로드
# ─────────────────────────────────────────────
FILE_PATH = "통합170901DATA취합.xlsx"

PASSWORD = "33"


with open(FILE_PATH, "rb") as f:
    office_file = msoffcrypto.OfficeFile(f)
    office_file.load_key(password=PASSWORD)
    
    decrypted = io.BytesIO()
    office_file.decrypt(decrypted)

wb = openpyxl.load_workbook(decrypted, read_only=True, data_only=True)
ws = wb["DATA"]
rows = list(ws.iter_rows(values_only=True))
            


In [3]:
columns = [
    "key", "영업담당자", "거래처명", "삼일모델명", "업체모델명", "CODE_No",
    "단가", "합의중량_g", "구단가_1",
    "구단가_2", "구단가_3", "구단가_4", "구단가_5","구단가_6" ,"구단가_7",
    "대표자", "사업자번호", "주소", "업태", "종목", "납품주소",
    "마감담당자_성명", "마감담당자_이메일", "마감담당자_연락처",
    "실무담당자_성명", "실무담당자_이메일", "실무담당자_연락처",
    "운임", "결제조건_일", "제품군", "비고"
]


data = [row for row in rows[2:] if any(row)]
df_raw = pd.DataFrame(data).iloc[:, : 31]
df_raw.columns = columns
df_raw = df_raw.drop(columns=["key"])
df_raw["거래처명"] = df_raw["거래처명"].astype(str).str.strip()
df_raw = df_raw[df_raw["거래처명"].notna() & (df_raw["거래처명"] != "None")].reset_index(drop=True)



In [4]:
# 거래처별 대표 정보 (첫 번째 등장 행 기준)
META_COLS = [
    "거래처명", "대표자", "사업자번호", "주소", "업태", "종목", "납품주소",
    "마감담당자_성명", "마감담당자_이메일", "마감담당자_연락처",
    "실무담당자_성명", "실무담당자_이메일", "실무담당자_연락처",
    "운임", "결제조건_일"
]
df_meta = df_raw[META_COLS].drop_duplicates(subset=["거래처명"]).set_index("거래처명")


In [5]:
# ─────────────────────────────────────────────
# 1. 고유 거래처명 수집 & 슬래시 분석
# ─────────────────────────────────────────────
all_clients = set(df_raw["거래처명"].unique())

slash_clients = {c for c in all_clients if "/" in c}
standalone_clients = {c for c in all_clients if "/" not in c}

# 슬래시 패턴에서 parent명 추출
parent_names_from_slash = {c.split("/")[0].strip() for c in slash_clients}

# 슬래시에만 있고 단독 행이 없는 parent → 자동 생성 필요
auto_create_parents = parent_names_from_slash - standalone_clients

print(f"전체 고유 거래처: {len(all_clients)}개")
print(f"  슬래시(/) 패턴: {len(slash_clients)}개")
print(f"  단독 행: {len(standalone_clients)}개")
print(f"  자동 생성할 Parent 행: {len(auto_create_parents)}개")


전체 고유 거래처: 900개
  슬래시(/) 패턴: 318개
  단독 행: 582개
  자동 생성할 Parent 행: 33개


In [6]:

# ─────────────────────────────────────────────
# 2. Master 테이블 구성
# ─────────────────────────────────────────────
records = []

# ── Step A: 단독 행 거래처 등록 ──────────────
for name in sorted(standalone_clients):
    is_parent = name in parent_names_from_slash  # 그냥 업체 이름들중에서 "/"리스트랑 비교해서 parent찾기
    meta = df_meta.loc[name].to_dict() if name in df_meta.index else {}
    records.append({
        "거래처명": name,
        "parent_명": None,          # 단독이므로 부모 없음
        "node_type": "고객사" if is_parent else "고객사",
        # 메타
        "대표자": meta.get("대표자"),  #메타df에서 가져오고
        "사업자번호": meta.get("사업자번호"),
        "주소": meta.get("주소"),
        "업태": meta.get("업태"),
        "종목": meta.get("종목"),
        "납품주소": meta.get("납품주소"),
        "마감담당자_성명": meta.get("마감담당자_성명"),
        "마감담당자_이메일": meta.get("마감담당자_이메일"),
        "마감담당자_연락처": meta.get("마감담당자_연락처"),
        "실무담당자_성명": meta.get("실무담당자_성명"),
        "실무담당자_이메일": meta.get("실무담당자_이메일"),
        "실무담당자_연락처": meta.get("실무담당자_연락처"),
        "운임": meta.get("운임"),
        "결제조건_일": meta.get("결제조건_일"),
    })



In [7]:
# ── Step B: 자동 생성 Parent 행 등록 ─────────
for name in sorted(auto_create_parents):
    records.append({
        "거래처명": name,
        "parent_명": None,
        "node_type": "고객사",
        "대표자": None, "사업자번호": None, "주소": None,
        "업태": None, "종목": None, "납품주소": None,
        "마감담당자_성명": None, "마감담당자_이메일": None, "마감담당자_연락처": None,
        "실무담당자_성명": None, "실무담당자_이메일": None, "실무담당자_연락처": None,
        "운임": None, "결제조건_일": None,
    })

In [8]:
# ── Step C: Child 행 등록 ─────────────────────
for name in sorted(slash_clients):
    parent_name = name.split("/")[0].strip()
    child_label = name.split("/", 1)[1].strip()
    meta = df_meta.loc[name].to_dict() if name in df_meta.index else {}
    records.append({
        "거래처명": child_label, # "/"slash포함된거하고싶음 "name" 넣으면됨
        "parent_명": parent_name,
        "node_type": "납품처",
        "대표자": meta.get("대표자"),
        "사업자번호": meta.get("사업자번호"),
        "주소": meta.get("주소"),
        "업태": meta.get("업태"),
        "종목": meta.get("종목"),
        "납품주소": meta.get("납품주소"),
        "마감담당자_성명": meta.get("마감담당자_성명"),
        "마감담당자_이메일": meta.get("마감담당자_이메일"),
        "마감담당자_연락처": meta.get("마감담당자_연락처"),
        "실무담당자_성명": meta.get("실무담당자_성명"),
        "실무담당자_이메일": meta.get("실무담당자_이메일"),
        "실무담당자_연락처": meta.get("실무담당자_연락처"),
        "운임": meta.get("운임"),
        "결제조건_일": meta.get("결제조건_일"),
    })

df_master = pd.DataFrame(records)


In [9]:
# ── 숫자 ID 부여 ──────────────────────────────
df_master = df_master.reset_index(drop=True)
df_master.insert(0, "거래처_ID", range(1, len(df_master) + 1)) #인덱스 0부터 끝까지 쫙넣어주기

In [10]:


# ── parent_id FK 연결 ─────────────────────────
name_to_id = df_master.set_index("거래처명")["거래처_ID"].to_dict()
df_master["parent_id"] = df_master["parent_명"].map(name_to_id) #인덱스 넘버로 붙이기

# parent_id를 두 번째 컬럼으로 이동
cols = df_master.columns.tolist()
cols.remove("parent_id")
cols.insert(2, "parent_id")
df_master = df_master[cols]

df_master


,거래처_ID,거래처명,parent_id,parent_명,node_type,대표자,사업자번호,주소,업태,종목,납품주소,마감담당자_성명,마감담당자_이메일,마감담당자_연락처,실무담당자_성명,실무담당자_이메일,실무담당자_연락처,운임,결제조건_일
0,1,(유)돌코리아,NaN,NaN,고객사,리차드웨인토만,219-81-28275,"서울특별시 강남구 영동대리511(삼성동,무역센터 트레이드타워12층1204호)","운수창고,도소매","창고업,과실및채소,음,식료품,가공식품,무역(농산물)",경기도 평택시 포승읍 평택항만길 305,윤민아,minayoon@dole.co.kr,010-8298-9424,정일성주임,NaN,010-2209-6748,None,40
1,2,DCT,NaN,NaN,고객사,이근수,126-25-23647,충북 진천군 덕산면 신척산단1로47,NaN,NaN,충북 진천군 덕산면 신척산단1로47,NaN,NaN,NaN,한태현선임,thhan@dctmaterial.com,010-4000-7911,None,30
2,3,DH글로벌,NaN,NaN,고객사,NaN,NaN,NaN,NaN,NaN,광주광역시 북구 첨단연신로 29번길 32,조영진,NaN,010-5719-5718,조영진,NaN,010-5719-5718,None,None
3,4,EPSKOREA,NaN,NaN,고객사,NaN,134-81-02673,경기 평택시 포승읍 평택항로 294,제조업,합성수지,경기 평택시 포승읍 평택항로 294,최현진,hyunjin9174@naver.com,010-9109-9174,최현진,hyunjin9174@naver.com,010-9109-9174,None,30
4,5,GS네트웍스,NaN,NaN,고객사,NaN,NaN,NaN,NaN,NaN,오산시 오산로149 5층 GS리테일 오산일배센터 E10번도크,NaN,NaN,NaN,NaN,NaN,010-8916-3961,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
928,929,서산,614.0,한성식품,납품처,NaN,130-81-44822,경기도 부천시 오정구 오정로 134번길 9-10(내동),"제조업,제조,도소매","과실채소절임식품, 김치, 고추, 반조미식품, 육가공식품, 식자재유통",경기도 안성시 신두만곡로 181,유일준차장,070-7596-3664,010-4067-7181,유일준과장,070-7596-3664,010-4067-7181,None,None
929,930,정선,614.0,한성식품,납품처,NaN,130-81-44822,경기도 부천시 오정구 오정로 134번길 9-10(내동),"제조업,제조,도소매","과실채소절임식품, 김치, 고추, 반조미식품, 육가공식품, 식자재유통",경기도 안성시 신두만곡로 181,신동명대리,NaN,010-8231-6446,신동명대리,NaN,010-8231-6446,None,None
930,931,효원,614.0,한성식품,납품처,NaN,301-81-88854,경기도 부천시 오정구 오정로 134번길 9-10(내동),"제조업,제조,도소매","과실채소절임식품, 김치, 고추, 반조미식품, 육가공식품, 식자재유통",경기도 안성시 신두만곡로 181,최광섭차장,NaN,010-8849-6835,최광섭차장,NaN,010-8849-6835,None,None
931,932,에프디시스,615.0,현대리바트,납품처,NaN,NaN,NaN,NaN,NaN,경기도 안성시 신두만곡로 181,NaN,NaN,NaN,NaN,NaN,NaN,None,None


In [11]:
# node_type 순서 정렬: parent/독립 먼저, child 뒤
#type_order = {"parent": 0, "독립": 1, "child": 2}
type_order = {"고객사": 0, "납품처": 1}

df_master["_sort"] = df_master["node_type"].map(type_order)
df_master = df_master.sort_values(["_sort", "거래처_ID"]).drop(columns=["_sort"]).reset_index(drop=True)
df_master["거래처_ID"] = range(1, len(df_master) + 1)
# parent_id 재연결 (정렬 후 ID 바뀌었으므로)
name_to_id = df_master.set_index("거래처명")["거래처_ID"].to_dict()
df_master["parent_id"] = df_master["parent_명"].map(name_to_id)

In [12]:
# ─────────────────────────────────────────────
# Step D: Duplicate 정리
# 같은 거래처명이 고객사 + 납품처 둘 다 있으면
# 납품처 행을 삭제하고, 그 ID를 가리키던 parent_id를 고객사 ID로 재연결
# ─────────────────────────────────────────────

# 고객사 중에서 납품처로도 존재하는 이름 찾기
고객사_names = set(df_master[df_master["node_type"] == "고객사"]["거래처명"])
납품처_names = set(df_master[df_master["node_type"] == "납품처"]["거래처명"])
duplicates = 고객사_names & 납품처_names

print(f"\n중복 발견: {len(duplicates)}개 → {sorted(duplicates)}")

for name in duplicates:
    # 진짜 ID = 고객사 행의 ID
    real_id = df_master.loc[
        (df_master["거래처명"] == name) & (df_master["node_type"] == "고객사"),
        "거래처_ID"
    ].values[0]

    # 가짜 ID = 납품처 행의 ID (삭제 대상)
    fake_id = df_master.loc[
        (df_master["거래처명"] == name) & (df_master["node_type"] == "납품처"),
        "거래처_ID"
    ].values[0]

    # fake_id를 parent_id로 가리키던 child들을 real_id로 재연결
    df_master.loc[df_master["parent_id"] == fake_id, "parent_id"] = real_id
    df_master.loc[df_master["parent_id"] == fake_id, "parent_명"] = name

    # 납품처 행 삭제
    df_master = df_master[
        ~((df_master["거래처명"] == name) & (df_master["node_type"] == "납품처"))
    ]

# ID 재부여 (삭제 후 빈 번호 정리)
df_master = df_master.reset_index(drop=True)
df_master["거래처_ID"] = range(1, len(df_master) + 1)

# parent_id도 새 ID 기준으로 재연결
name_to_id = df_master.set_index("거래처명")["거래처_ID"].to_dict()
df_master["parent_id"] = df_master["parent_명"].map(name_to_id)

print(f"정리 후 전체: {len(df_master)}개")


중복 발견: 21개 → ['건어맨', '겸엑스', '더힐링팜', '동아에스티', '디베랴', '로지스프로', '메이플델리', '명정보기술', '바이오플레이', '성신푸드', '아이탐푸드', '우진수산', '제이월드텍', '지웅', '태성산업', '태진패키지', '팀프레시', '파스토', '피시원', '한국이노팩', '히즈테이블']
정리 후 전체: 911개


In [13]:
# ─────────────────────────────────────────────
# 4. 결과 요약
# ─────────────────────────────────────────────
parents   = df_master[df_master["node_type"] == "고객사"]
children  = df_master[df_master["node_type"] == "납품처"]

print(f"\n{'='*55}")
print("【Master 테이블 계층 구조 요약】")
print(f"  Parent  (본사/주관사)  : {len(parents):>4}개")
print(f"  Child   (지점/납품처)  : {len(children):>4}개")
print(f"  전체                  : {len(df_master):>4}개")
print(f"{'='*55}")

print("\n【Parent → Child 샘플】")
for p_name in ["청호나이스", "CJ대한통운", "동원홈푸드", "한국파렛트풀", "컬리"]:
    if p_name not in name_to_id:
        continue
    pid = name_to_id[p_name]
    kids = df_master[df_master["parent_id"] == pid][["거래처_ID", "거래처명"]].values
    print(f"  [{pid}] {p_name}")
    for kid_id, kid_name in kids[:5]:
        print(f"       └─ [{kid_id}] {kid_name}")
    if len(kids) > 5:
        print(f"       └─ ... 외 {len(kids)-5}개")


【Master 테이블 계층 구조 요약】
  Parent  (본사/주관사)  :  615개
  Child   (지점/납품처)  :  296개
  전체                  :  911개

【Parent → Child 샘플】
  [473] 청호나이스
       └─ [850] 미래씨엔에이치
       └─ [851] 에스엠나이스
       └─ [852] 협신산업
  [585] CJ대한통운
       └─ [621] 가미락
       └─ [622] 강원도옥수수총각
       └─ [623] 그림엘푸드
       └─ [624] 남천영농조합법인
       └─ [625] 도당
       └─ ... 외 21개
  [588] 동원홈푸드
       └─ [672] 가산공장
       └─ [673] 금천미트(대전)
       └─ [674] 대전센터
       └─ [675] 시화센터
       └─ [676] 장호원
       └─ ... 외 1개
  [613] 한국파렛트풀
       └─ [887] 대상
       └─ [888] 더다냉동물류
       └─ [889] 더유리빙
       └─ [890] 동원로엑스
       └─ [891] 두레냉동
       └─ ... 외 14개
  [607] 컬리
       └─ [853] 남양주
       └─ [854] 송파


In [14]:
df_master[df_master['거래처명'].duplicated(keep=False)]['거래처명'].to_list()

['덕평',
 '송파',
 '이천',
 '한일전기',
 '이천',
 '김포',
 '광주',
 '인천',
 '남선푸드',
 '진원',
 '광주',
 '인천',
 '남선푸드',
 '안성',
 '한일전기',
 '송파',
 '진원',
 '라라스윗',
 '송파',
 '덕평',
 '수원',
 '이천',
 '광주',
 '김포',
 '수원',
 '안성',
 '인천',
 '라라스윗']

In [16]:
df_master[df_master['거래처명'] == '한일전기']

,거래처_ID,거래처명,parent_id,parent_명,node_type,대표자,사업자번호,주소,업태,종목,납품주소,마감담당자_성명,마감담당자_이메일,마감담당자_연락처,실무담당자_성명,실무담당자_이메일,실무담당자_연락처,운임,결제조건_일
664,665,한일전기,34.0,강원공장,납품처,NaN,NaN,NaN,NaN,NaN,경기도 안성시 신두만곡로 181,NaN,NaN,NaN,NaN,NaN,NaN,None,None
799,800,한일전기,370.0,용인공장,납품처,NaN,NaN,NaN,NaN,NaN,경기도 안성시 신두만곡로 181,NaN,NaN,NaN,NaN,NaN,NaN,None,None


In [224]:
df_master[df_master['parent_명'] == '용인공장']

,거래처_ID,거래처명,parent_id,parent_명,node_type,대표자,사업자번호,주소,업태,종목,납품주소,마감담당자_성명,마감담당자_이메일,마감담당자_연락처,실무담당자_성명,실무담당자_이메일,실무담당자_연락처,운임,결제조건_일
791,792,KCP이천,370.0,용인공장,납품처,NaN,NaN,NaN,NaN,NaN,경기도 안성시 신두만곡로 181,NaN,NaN,NaN,NaN,NaN,010-9265-5081,None,None
792,793,SHS,370.0,용인공장,납품처,NaN,NaN,NaN,NaN,NaN,경기도 안성시 신두만곡로 181,NaN,NaN,NaN,NaN,NaN,010-9265-5081,None,None
793,794,새빛공조,370.0,용인공장,납품처,NaN,NaN,NaN,NaN,NaN,경기도 안성시 신두만곡로 181,NaN,NaN,NaN,NaN,NaN,NaN,None,None
794,795,써모렙코리아,370.0,용인공장,납품처,홍종욱,372-81-00976,"서울 강남구 테헤란로79길6, 9층",서비스,소프트웨어 개발 및 서비스,경기도 안성시 신두만곡로 181,NaN,NaN,NaN,유형근 센터장,NaN,010-4143-6697,None,None
795,796,온세물류,370.0,용인공장,납품처,NaN,NaN,NaN,NaN,NaN,경기도 안성시 신두만곡로 181,NaN,NaN,NaN,NaN,NaN,NaN,None,None
796,797,우성AFC,370.0,용인공장,납품처,NaN,NaN,NaN,NaN,NaN,경기도 안성시 신두만곡로 181,NaN,NaN,NaN,NaN,NaN,NaN,None,None
797,798,유한산업,370.0,용인공장,납품처,NaN,NaN,NaN,NaN,NaN,경기도 안성시 신두만곡로 181,NaN,010-8836-8199,NaN,NaN,NaN,NaN,None,None
798,799,한국파렛트풀/유한산업,370.0,용인공장,납품처,NaN,NaN,NaN,NaN,NaN,경기도 안성시 신두만곡로 181,NaN,NaN,NaN,NaN,NaN,NaN,None,None
799,800,한일전기,370.0,용인공장,납품처,NaN,NaN,NaN,NaN,NaN,경기도 안성시 신두만곡로 181,NaN,NaN,NaN,NaN,NaN,NaN,None,None
800,801,휴앤로지스,370.0,용인공장,납품처,NaN,NaN,NaN,NaN,NaN,경기도 안성시 신두만곡로 181,NaN,NaN,NaN,NaN,NaN,010-3333-7969,None,None
